In [20]:
import pandas as pd
import requests
import json


In [2]:
#Website were puklling the endpouint from to get the data 
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

response = requests.get(url, params={"$limit": 5}, timeout=60)
response.raise_for_status()

data = response.json()
df = pd.DataFrame(data)

display(df)

,unique_key,created_date,agency,agency_name,complaint_type,descriptor,location_type,status,community_board,police_precinct,...,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,council_district,bbl,descriptor_2
0,70765514,2026-10-16T01:51:34.000,NYPD,New York City Police Department,Noise - Commercial,Loud Music/Party,Store/Commercial,Unspecified,0 Unspecified,Unspecified,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,70765005,2026-10-16T01:51:28.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,In Progress,01 QUEENS,Precinct 114,...,34 AVENUE,35 AVENUE,34 AVENUE,35 AVENUE,ADDRESS,ASTORIA,12 STREET,26,4005230002,NaN
2,70770337,2026-10-16T01:51:27.000,NYPD,New York City Police Department,Noise - Residential,Loud Talking,Residential Building/House,In Progress,0 Unspecified,Unspecified,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,70769727,2026-10-16T01:51:23.000,DSNY,Department of Sanitation,Residential Disposal Complaint,Storage Area Not Provided,Sidewalk,In Progress,0 Unspecified,Unspecified,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Super or Building Manager
4,70766819,2026-10-16T01:51:15.000,DSNY,Department of Sanitation,Dirty Condition,Trash,Sidewalk,In Progress,0 Unspecified,Unspecified,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Littering


In [ ]:
#Filter for street conditions only using parameters DOT( department of transportation) and 
#only street condition issues 
#Only pulling the first 5 in this request.get
params = {
    "$where": "agency = 'DOT' AND complaint_type = 'Street Condition'",
    "$limit": 5
}

response = requests.get(url, params=params, timeout=60)



df = pd.DataFrame(response.json())

display(df[["unique_key", "agency", "complaint_type", "descriptor"]])



,unique_key,agency,complaint_type,descriptor
0,69177427,DOT,Street Condition,Pothole
1,69185057,DOT,Street Condition,Pothole
2,68168575,DOT,Street Condition,Pothole
3,68179684,DOT,Street Condition,Pothole
4,68176123,DOT,Street Condition,Pothole


In [4]:
#Collecting time stamps from January 20205 

params = {
    "$where": (
        "agency = 'DOT' "
        "AND complaint_type = 'Street Condition' "
        "AND created_date >= '2025-01-01T00:00:00' "
        "AND created_date < '2025-02-01T00:00:00'"
    ),
    "$limit": 5
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

df = pd.DataFrame(response.json())

display(df[["created_date", "complaint_type", "descriptor"]])

,created_date,complaint_type,descriptor
0,2025-01-01T02:27:33.000,Street Condition,Pothole
1,2025-01-01T02:31:08.000,Street Condition,Pothole
2,2025-01-01T02:39:44.000,Street Condition,Dumpster - Construction Waste
3,2025-01-01T02:55:52.000,Street Condition,Pothole
4,2025-01-01T04:20:00.000,Street Condition,Wear & Tear


In [ ]:
#matching requests in January total amount of complaints in January 
count_params = {
    "$where": params["$where"],
    "$select": "count(*) AS total_requests"
}

response = requests.get(url, params=count_params, timeout=60)
response.raise_for_status()

display(pd.DataFrame(response.json()))

,total_requests
0,5474


In [ ]:
#Printing the row and column amount 
collect_params = {
    "$where": params["$where"],
    "$order": "created_date ASC, unique_key ASC",
    "$limit": 6000
}

response = requests.get(url, params=collect_params, timeout=120)
response.raise_for_status()

january_df = pd.DataFrame(response.json())

print("Rows:", january_df.shape[0])
print("Columns:", january_df.shape[1])

Rows: 5474
Columns: 36


In [7]:
#Checking for duplicates 

print("Duplicate request IDs:",
      january_df["unique_key"].duplicated().sum())

print("\nColumn names:")
print(january_df.columns.tolist())

Duplicate request IDs: 0

Column names:
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'address_type', 'city', 'facility_type', 'status', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'police_precinct', 'borough', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'intersection_street_1', 'intersection_street_2', 'council_district', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'latitude', 'longitude', 'location', 'descriptor_2', 'location_type', 'landmark', 'bbl']


In [9]:
#Calculate missing values 
display(january_df.isna().sum())

missing_closure = january_df[
    january_df["closed_date"].isna()
]

display(missing_closure["status"].value_counts())

unique_key                           0
created_date                         0
closed_date                         85
agency                               0
agency_name                          0
complaint_type                       0
descriptor                           0
incident_zip                        68
incident_address                  1550
street_name                       1550
cross_street_1                    1572
cross_street_2                    1571
address_type                         0
city                               566
facility_type                     1723
status                               0
resolution_description               3
resolution_action_updated_date       3
community_board                      0
police_precinct                      0
borough                              0
open_data_channel_type               0
park_facility_name                   0
park_borough                         0
intersection_street_1             2184
intersection_street_2    

status
Pending    85
Name: count, dtype: int64

In [11]:
#Convert text into actual dates we can use 

january_df["created_date"] = pd.to_datetime(
    january_df["created_date"]
)

january_df["closed_date"] = pd.to_datetime(
    january_df["closed_date"]
)

display(january_df[["created_date", "closed_date"]].dtypes)

created_date    datetime64[us]
closed_date     datetime64[us]
dtype: object

In [14]:
#Calculating time difference between openand closure, alongside negative closure times
time_difference = (
    january_df["closed_date"] - january_df["created_date"]
)

january_df["closure_hours"] = (
    time_difference.dt.total_seconds() / 3600
)

display(
    january_df[["created_date", "closed_date", "closure_hours"]].head(10)
)

#check for negative times 
negative_times = january_df["closure_hours"] < 0

print("Negative closure times:", negative_times.sum())

display(january_df["closure_hours"].describe())


,created_date,closed_date,closure_hours
0,2025-01-01 02:27:33,2025-01-02 12:40:00,34.207500
1,2025-01-01 02:31:08,2025-01-06 10:20:00,127.814444
2,2025-01-01 02:39:44,2025-01-02 11:30:02,32.838333
3,2025-01-01 02:55:52,2025-01-06 11:00:00,128.068889
4,2025-01-01 04:20:00,NaT,NaN
5,2025-01-01 04:28:31,2025-01-02 12:03:27,31.582222
6,2025-01-01 06:00:17,2025-01-02 20:10:00,38.161944
7,2025-01-01 06:01:21,2025-01-02 20:45:00,38.727500
8,2025-01-01 06:02:48,2025-01-02 21:35:00,39.536667
9,2025-01-01 06:03:33,2025-01-02 22:05:00,40.024167


Negative closure times: 0


count     5389.000000
mean        50.481935
std        321.988083
min          0.000000
25%         18.547222
50%         23.613333
75%         42.217222
max      12122.434444
Name: closure_hours, dtype: float64

In [ ]:
#Explanation for the mean being 50 hours since the largest time differrences span into 2026 
#May reflect administration oversight 
display(
    january_df.sort_values("closure_hours", ascending=False)[
        ["created_date", "closed_date", "descriptor", "status", "closure_hours"]
    ].head(5)
)

,created_date,closed_date,descriptor,status,closure_hours
2552,2025-01-14 08:43:53,2026-06-03 11:09:57,Cave-in,Closed,12122.434444
674,2025-01-04 23:12:14,2026-01-07 11:00:00,Pothole,Closed,8819.796111
155,2025-01-01 15:13:35,2026-01-02 11:25:00,Pothole,Closed,8780.190278
2583,2025-01-14 13:01:02,2026-01-14 16:25:00,Pothole,Closed,8763.399444
1314,2025-01-07 16:09:48,2025-12-02 09:18:15,Blocked - Construction,Closed,7889.140833


In [16]:
#Our target variable (binary classification over the 48 hour mark or not)

over_48_hours = january_df["closure_hours"] > 48

print("Recorded closures after 48 hours:", over_48_hours.sum())

Recorded closures after 48 hours: 1138


In [19]:
#copy of january data
january_backup = january_df.copy()
data_folder = Path("../data")
data_folder.mkdir(exist_ok=True)

january_backup.to_csv(
    data_folder / "street_conditions_january_2025.csv",
    index=False
)

print("Data saved!")

Data saved!


In [22]:

with open("../data/street_conditions_january_2025_raw.json", "w") as file:
    json.dump(response.json(), file, indent=2)

print(len(response.json()))
print("Raw JSON saved!")

5474
Raw JSON saved!
